# Pandas para contadores — Ejercicios**COMERCIALIZADORA ANDINA S.A.S.** (datos simulados) · Periodo enero–marzo 202615 ejercicios progresivos sobre datos contables colombianos: balance de prueba con PUC,libro auxiliar, facturas de proveedores con retenciones y extracto bancario.**Cómo trabajar este cuaderno**1. Lee el enunciado, escribe tu código en la celda `TODO` y ejecútala con `Shift + Enter`.2. Si te trabas, mira la solución en `02_soluciones.ipynb` — pero inténtalo primero.3. No pasa nada si rompes algo: vuelve a ejecutar desde el Ejercicio 1.**Antes de empezar**: en VS Code, abre este archivo, arriba a la derecha elige elkernel de Python y asegúrate de haber corrido `pip install pandas openpyxl jupyter`.---## Cómo se ejecuta un notebook| Atajo | Qué hace ||---|---|| `Shift + Enter` | ejecuta la celda y pasa a la siguiente || `Ctrl + Enter` | ejecuta la celda y se queda ahí || `Esc` + `A` / `B` | inserta celda arriba / abajo || `Esc` + `D D` | borra la celda |El orden importa: si no ejecutaste el Ejercicio 1, las variables `aux`, `bp`, `ext` y `fac`no existen y todo lo demás falla con `NameError`.---

## Ejercicio 1 — Leer los archivosPrimer contacto con `pandas`. Un **DataFrame** es una tabla; una **Series** es una columna.Carga los cuatro archivos de la carpeta `datos/` y muestra, de cada uno, cuántas filas y columnas tiene.> **Ojo profesional:** `codigo_puc` y los NIT son *códigos*, no números. Si dejas que pandas los lea como enteros pierdes los ceros a la izquierda y no podrás cruzarlos después. Usa `dtype={"codigo_puc": str}`.

In [ ]:
import pandas as pdRUTA = "datos/"# TODO: leer los 4 archivos con pd.read_csvaux = ...   # libro_auxiliar.csv   -> codigo_puc y nit_tercero como textobp  = ...   # balance_prueba.csv   -> codigo_puc como textoext = ...   # extracto_bancario.csvfac = ...   # facturas_proveedores.csv -> nit_proveedor como texto# TODO: imprimir el .shape de cada uno

## Ejercicio 2 — Radiografía del libro auxiliarAntes de calcular nada hay que saber **qué tipo de dato** tiene cada columna. Usa:- `.info()` — tipos y nulos- `.dtypes` — solo tipos- `.describe()` — estadísticos de las columnas numéricas- `.columns` — nombres de columnasAdemás convierte `fecha` a tipo fecha real con `pd.to_datetime`, porque leída del CSV llega como texto.

In [ ]:
# TODO: aux.info()# TODO: convertir la columna fecha a datetime# TODO: aux[["debito", "credito"]].describe()

## Ejercicio 3 — Partida doble: ¿cuadra el auxiliar?La primera validación de cualquier archivo que te entreguen: **suma de débitos = suma de créditos**.Calcula ambas sumas, su diferencia, y escribe una condición que imprima `CUADRA` o `NO CUADRA`.> Redondea a 2 decimales antes de comparar: los flotantes arrastran error binario y `0.1 + 0.2 != 0.3` en Python.

In [ ]:
# TODO: total_debitos, total_creditos, diferencia# TODO: imprimir el diagnostico

## Ejercicio 4 — Seleccionar y filtrarTres formas de acceder a los datos:| Qué quieres | Cómo ||---|---|| una columna | `df["debito"]` || varias columnas | `df[["fecha", "debito"]]` || filas por condición | `df[df["codigo_puc"] == "111005"]` || fila/columna por etiqueta | `df.loc[fila, "columna"]` || fila/columna por posición | `df.iloc[0, 3]` |Extrae todos los movimientos de la cuenta **111005 (banco)**, cuenta cuántos son y calcula el movimiento neto (débitos − créditos).

In [ ]:
# TODO: banco = filtrar aux por codigo_puc == "111005"# TODO: cuantos movimientos y cual es el neto

## Ejercicio 5 — Filtros compuestosSe combinan con `&` (y), `|` (o), `~` (no), y **cada condición va entre paréntesis**.Encuentra los movimientos que cumplan las tres cosas:1. son de compras — el `comprobante` empieza por `FC`2. son de la cuenta de inventario `143530`3. el débito supera $3.000.000Pistas: `.str.startswith("FC")`, y para el mes `aux["fecha"].dt.month`.

In [ ]:
# TODO: filtro de tres condicionescompras_grandes = ...

## Ejercicio 6 — Limpieza de las facturas de proveedoresEl archivo `facturas_proveedores.csv` viene sucio, como en la vida real. Corrige:1. **Espacios y mayúsculas** en `proveedor` → `.str.strip()` y `.str.upper()`2. **Filas duplicadas** → `.duplicated().sum()` y `.drop_duplicates()`3. **Nulos** en `estado` → `.isna().sum()` y `.fillna("PENDIENTE")`4. **Fechas** en formato `dd/mm/yyyy` → `pd.to_datetime(..., format="%d/%m/%Y")`Trabaja sobre una copia (`fac.copy()`) para no dañar el original.

In [ ]:
fac_limpio = fac.copy()# TODO: 1) normalizar proveedor# TODO: 2) eliminar duplicados# TODO: 3) rellenar estado nulo# TODO: 4) convertir fecha_factura a datetime

## Ejercicio 7 — Recalcular las retencionesNunca confíes en las retenciones que trae el archivo: recalcúlalas.Sobre `fac_limpio` crea:- `rf_calc` = 2,5 % de la base gravable (compras generales)- `ica_calc` = 9,66 × 1.000 de la base gravable- `dif_rf` y `dif_ica` = diferencia contra lo registrado- una columna `alerta` que diga `OK` o `REVISAR` cuando la diferencia absoluta supere $1Luego muestra solo las filas con alerta.

In [ ]:
# TODO: columnas calculadas y diferencias# TODO: columna alerta con np.where o con una comparacion booleana

## Ejercicio 8 — groupby: ventas por mes`groupby` es el equivalente a una tabla dinámica. La receta es siempre la misma:```pythondf.groupby("columna_que_agrupa")["columna_a_sumar"].sum()```Crea una columna `mes` (`aux["fecha"].dt.to_period("M")`) y calcula las **ventas por mes**: créditos de la cuenta `413595`.

In [ ]:
# TODO: columna mes# TODO: ventas por mes de la cuenta 413595

## Ejercicio 9 — Top de clientes y de proveedoresCon `groupby` + `sort_values` + `head` sacas cualquier ranking.1. Top 5 **clientes** por ventas (créditos de `413595` agrupados por `nombre_tercero`)2. Top 5 **proveedores** por compras (débitos de `143530` en comprobantes `FC`)3. Para el top de clientes agrega la columna `participacion` en % sobre el total.

In [ ]:
# TODO: top clientes# TODO: top proveedores# TODO: participacion %

## Ejercicio 10 — pivot_table: gastos por centro de costo y mes`pivot_table` arma la tabla dinámica con filas, columnas y totales:```pythondf.pivot_table(index=..., columns=..., values=..., aggfunc="sum", margins=True)```Filtra las cuentas de gasto (las que empiezan por `5` — usa `.str.startswith("5")`) y arma la tabla con centro de costo en filas, mes en columnas y débitos como valor, con totales.

In [ ]:
# TODO: filtrar gastos (cuentas clase 5)# TODO: pivot_table con margins=True

## Ejercicio 11 — Reconstruir el balance de prueba desde el auxiliarEste es el cruce que hace un auditor: el balance que te entregan **debe** poder reconstruirse desde el auxiliar.1. Agrupa el auxiliar por `codigo_puc` sumando débitos y créditos2. Cruza (`merge`) ese resultado con `bp` por `codigo_puc`3. Calcula `dif_debitos` y `dif_creditos`4. Muestra las cuentas donde la diferencia no sea ceroSobre `merge`: `how="outer"` te deja ver también las cuentas que están en un archivo y no en el otro — justo lo que quieres detectar.

In [ ]:
# TODO: mov = groupby del auxiliar# TODO: cruce = merge con bp# TODO: diferencias

## Ejercicio 12 — La ecuación contableLa clase de la cuenta es el primer dígito del PUC: `1` activo, `2` pasivo, `3` patrimonio, `4` ingreso, `5` gasto, `6` costo.1. Crea en `bp` la columna `clase` con `.str[0]`2. Suma `saldo_final` por clase (recuerda: en este archivo los saldos crédito vienen en negativo)3. Verifica que **Activo = Pasivo + Patrimonio + Resultado del ejercicio**Pista: si los saldos crédito son negativos, la suma de *todos* los saldos finales debe dar cero.

In [ ]:
# TODO: columna clase# TODO: resumen por clase# TODO: verificacion de la ecuacion

## Ejercicio 13 — Estado de resultados del trimestreArma el P&G con las clases 4, 6 y 5:| Renglón | Fórmula ||---|---|| Ingresos operacionales | créditos 41 − débitos 41 || Costo de ventas | clase 6 || **Utilidad bruta** | Ingresos − Costo || Gastos de administración y ventas | clase 5 || **Utilidad operacional** | Utilidad bruta − Gastos || Margen bruto % / Margen operacional % | sobre ingresos |Constrúyelo como un DataFrame de dos columnas (`concepto`, `valor`) para poder exportarlo después.

In [ ]:
# TODO: calcular cada renglon desde bp o desde aux# TODO: armar el DataFrame del estado de resultados

## Ejercicio 14 — Conciliación bancariaEl ejercicio completo. En libros, el banco es la cuenta `111005`; en el extracto, cada fila trae `referencia` (el comprobante) y `valor`.1. Movimientos de banco en libros con su valor neto (`debito - credito`)2. Movimientos del extracto agrupados por `referencia`3. `merge` con `how="outer"` e `indicator=True`4. Clasifica las partidas conciliatorias:   - **solo en libros** → consignaciones o cheques pendientes   - **solo en extracto** → GMF, comisiones, rendimientos, notas débito   - **en ambos con diferencia de valor** → error de digitación5. Prueba el saldo: `saldo extracto + partidas solo en libros − partidas solo en extracto = saldo en libros`

In [ ]:
# TODO: libros_banco (neto por comprobante)# TODO: extracto agrupado por referencia# TODO: merge outer con indicator# TODO: clasificar partidas y probar el saldo

## Ejercicio 15 — Exportar el informe a ExcelTodo el trabajo termina en un entregable. Con `pd.ExcelWriter` escribes varias hojas en un mismo archivo:```pythonwith pd.ExcelWriter("informe.xlsx", engine="openpyxl") as w:    df1.to_excel(w, sheet_name="Hoja1", index=False)    df2.to_excel(w, sheet_name="Hoja2", index=False)```Exporta a `salidas/informe_contable.xlsx` cuatro hojas: balance por clase, estado de resultados, partidas conciliatorias y facturas con alerta.

In [ ]:
# TODO: crear la carpeta salidas y exportar el informe con varias hojas

---## Retos para seguir practicando1. **Cartera por edades**: clasifica el saldo de clientes (`130505`) en 0–30, 31–60, 61–90 y +90 días   usando `pd.cut` sobre los días transcurridos desde la fecha del documento.2. **Certificados de retención**: agrupa la retefuente practicada (`236540`) por NIT y por mes,   y genera un archivo por proveedor con `to_excel` dentro de un `for`.3. **Formato 1001 (información exógena)**: arma un DataFrame con NIT, concepto, pago acumulado   y retención practicada por tercero para el trimestre.4. **Indicadores**: calcula razón corriente, capital de trabajo y rotación de inventarios   con los saldos del balance.5. **Gráfico**: `ventas_mes.plot(kind="bar")` — necesitas `pip install matplotlib`.## Chuleta de pandas| Necesito | Código ||---|---|| leer / escribir | `pd.read_csv()` · `pd.read_excel()` · `df.to_excel()` || ver | `.head()` `.tail()` `.info()` `.describe()` `.shape` `.columns` || filtrar | `df[df["col"] > 0]` · `.isin([...])` · `.between(a, b)` · `.query("col > 0")` || texto | `.str.strip()` `.str.upper()` `.str.contains()` `.str.startswith()` || fechas | `pd.to_datetime()` · `.dt.month` `.dt.year` `.dt.to_period("M")` || nulos | `.isna().sum()` · `.fillna(0)` · `.dropna()` || agrupar | `.groupby("col")["val"].sum()` · `.agg(["sum", "count"])` · `.pivot_table()` || cruzar | `.merge(otro, on="llave", how="left/outer", indicator=True)` || ordenar | `.sort_values("col", ascending=False)` · `.nlargest(5, "col")` || nueva columna | `df["nueva"] = ...` · `.assign(nueva=...)` · `np.where(cond, a, b)` |## Si algo falla| Error | Causa habitual ||---|---|| `FileNotFoundError` | VS Code no está parado en la carpeta del proyecto — usa *Abrir carpeta*, no *Abrir archivo* || `NameError` | no ejecutaste la celda anterior || `KeyError: 'columna'` | el nombre no existe: revisa `df.columns` || `SettingWithCopyWarning` | estás modificando un filtro; usa `.copy()` || suma que no cuadra por centavos | falta `.round(2)` antes de comparar |